In [ ]:
import pandas as pd

# ----------------------------
# 1. LOAD + PREPROCESS
# ----------------------------
from src.data.fetch_data import load_all_raw
from src.data.preprocess import preprocess_all

raw = load_all_raw()
processed = preprocess_all(raw)

# ----------------------------
# 2. RUN ALL MODELS (TRAIN + FUTURE FC ONLY)
# ----------------------------
from src.models.arima import run_arima_pipeline
from src.models.prophet_model import run_prophet_pipeline
from src.models.lstm import run_lstm_pipeline
from src.models.gru import run_gru_pipeline
from src.models.transformer import run_transformer_pipeline

arima_preds, arima_fc, arima_metrics = run_arima_pipeline(processed, n_forecast=2)
prophet_preds, prophet_fc, prophet_metrics = run_prophet_pipeline(processed)
lstm_preds, lstm_fc, lstm_metrics = run_lstm_pipeline(processed, epochs=30)
gru_preds, gru_fc, gru_metrics = run_gru_pipeline(processed, epochs=30)
trans_preds, trans_fc, trans_metrics = run_transformer_pipeline(processed, epochs=30)

# ----------------------------
# 3. BUILD FINAL FORECAST TABLE (NEXT 2 DAYS)
# ----------------------------
tickers = list(processed.keys())

final_forecasts = {}

for ticker in tickers:
    final_forecasts[ticker] = pd.DataFrame({
        "ARIMA": arima_fc[ticker][:2],
        "Prophet": prophet_fc[ticker][:2],
        "LSTM": lstm_fc[ticker][:2],
        "GRU": gru_fc[ticker][:2],
        "Transformer": trans_fc[ticker][:2],
    })

# ----------------------------
# 4. ASSIGN FUTURE DATES (AFTER DATA END)
# ----------------------------
from pandas.tseries.offsets import BDay

last_date = processed[tickers[0]]["close"].index[-1]
future_dates = pd.date_range(start=last_date + BDay(1), periods=2, freq="B")

# attach dates
for ticker in final_forecasts:
    final_forecasts[ticker].index = future_dates

# ----------------------------
# 5. DISPLAY RESULT FOR ONE STOCK (EXAMPLE)
# ----------------------------
final_forecasts["TCS.NS"]